# 📝 Week 5: Keyword Extraction pada Dataset MBG

Notebook ini membahas proses ekstraksi kata kunci dari kumpulan artikel berita Makanan Bergizi Gratis (MBG) menggunakan pendekatan **TF-IDF** dan **N-gram Analysis**.

## 🎯 Tujuan Pembelajaran:
1. Memuat dan memahami struktur dataset artikel MBG
2. Melakukan preprocessing teks bahasa Indonesia
3. Mengekstrak kata kunci menggunakan TF-IDF
4. Menganalisis pola frasa penting menggunakan N-gram (bigram & trigram)

## 📦 Instalasi Dependencies

In [2]:
import warnings
warnings.filterwarnings('ignore')

%pip install -q nltk scikit-learn pandas openpyxl

print("✅ Semua library berhasil diinstall!")

Note: you may need to restart the kernel to use updated packages.
✅ Semua library berhasil diinstall!


## 📚 Import Libraries

In [3]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print("✅ Import libraries berhasil!")

✅ Import libraries berhasil!


---

## 1️⃣ Load Dataset

Dataset berisi artikel berita tentang program **Makanan Bergizi Gratis (MBG)** yang dikumpulkan dari berbagai media Indonesia.

File Excel terdiri dari dua sheet:
- **`Case`** — artikel kasus/insiden terkait MBG
- **`Main`** — artikel utama tentang program MBG

| Kolom | Deskripsi |
|---|---|
| ID | Nomor unik artikel |
| Judul | Judul berita |
| Konten | Isi artikel lengkap |
| Kategori | Kategori berita |
| Sumber | Media online sumber |
| Tanggal | Tanggal terbit |
| Tone | Nada berita (positif/negatif/netral) |

In [5]:
# Load dataset dari file Excel
file_path = 'Dataset MBG.xlsx'

df_case = pd.read_excel(file_path, sheet_name='Case')
df_main = pd.read_excel(file_path, sheet_name='Main')

print("✅ Dataset berhasil dimuat!")
print(f"Sheet 'Case'  : {df_case.shape[0]} artikel, {df_case.shape[1]} kolom")
print(f"Sheet 'Main'  : {df_main.shape[0]} artikel, {df_main.shape[1]} kolom")

✅ Dataset berhasil dimuat!
Sheet 'Case'  : 11 artikel, 3 kolom
Sheet 'Main'  : 32 artikel, 9 kolom


In [6]:
# Preview sheet Main
df_main.head(3)

,Case ID,Case,Newsroom,Tanggal,Judul,Konten,Link,Kota,Tone
0,C01,Cara Pelaksanaan di Lapangan,Kompas.com\t,2024-11-01,Persiapan Makan Bergizi Gratis: Sudah Simulasi...,"JAKARTA, KOMPAS.com - Pemerintah terus menyiap...",Persiapan Makan Bergizi Gratis: Sudah Simulasi...,Jakarta,Netral
1,C02,Proyeksi dan Alokasi Anggaran,Kompas.com\t,2025-08-13,Sri Mulyani Ungkap Anggaran Program MBG Lebih ...,"JAKARTA, KOMPAS.com - Menteri Keuangan Sri Mul...",Sri Mulyani Ungkap Anggaran Program MBG Lebih ...,Jakarta,Netral
2,C01,Cara Pelaksanaan di Lapangan,Kompas.com\t,2025-08-13,Target 20 Juta Penerima Makan Bergizi Gratis D...,KOMPAS.com – Kepala Badan Gizi Nasional (BGN) ...,Target 20 Juta Penerima Makan Bergizi Gratis D...,Jakarta,Netral


In [7]:
# Preview sheet Case
df_case.head(3)

,Case ID,Case,Penjelasan
0,C01,Cara Pelaksanaan di Lapangan,"Berita tentang bagaimana makanan disiapkan, di..."
1,C02,Pembahasan Soal Anggaran & Dana,Berita yang membahas semua hal tentang uang: b...
2,C03,Tujuan Program & Siapa Penerimanya,Berita yang menjelaskan kenapa program ini ada...


---

## 2️⃣ Preprocessing Teks

Sebelum ekstraksi kata kunci, teks perlu diproses agar lebih bersih dan terstandarisasi:

### 2.1 Fungsi Preprocessing

In [8]:
# Stopwords bahasa Indonesia
stop_words = set(stopwords.words('indonesian'))

def preprocess_text(text):
    """Lowercase, hapus tanda baca & angka, tokenisasi, hapus stopwords."""
    if pd.isna(text):
        return ''
    # Lowercase
    text = text.lower()
    # Hapus tanda baca dan angka
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenisasi
    tokens = word_tokenize(text)
    # Hapus stopwords
    tokens = [w for w in tokens if w not in stop_words and len(w) > 1]
    return ' '.join(tokens)

print("✅ Fungsi preprocessing siap!")

✅ Fungsi preprocessing siap!


### 2.2 Terapkan Preprocessing

In [11]:
# Terapkan preprocessing ke kolom Konten
df_main['Konten_Clean'] = df_main['Konten'].apply(preprocess_text)
df_case['Konten_Clean'] = df_case['Penjelasan'].apply(preprocess_text)

print("✅ Preprocessing selesai!")
print("\nContoh teks asli (30 kata pertama):")
print(df_main['Konten'].iloc[0][:200])
print("\nContoh teks setelah preprocessing (30 token):")
print(' '.join(df_main['Konten_Clean'].iloc[0].split()[:30]))

✅ Preprocessing selesai!

Contoh teks asli (30 kata pertama):
JAKARTA, KOMPAS.com - Pemerintah terus menyiapkan dan menguji coba program makan bergizi gratis yang rencananya akan dimulai pada 2 Januari 2025. Terbaru, pemerintah memiliki opsi untuk mengganti susu

Contoh teks setelah preprocessing (30 token):
jakarta kompascom pemerintah menguji coba program makan bergizi gratis rencananya januari terbaru pemerintah memiliki opsi mengganti susu kemasan susu cair alternatif pengganti menu program makan bergizi gratis menteri sekretaris negara


---

## 3️⃣ Ekstraksi Kata Kunci dengan TF-IDF

**TF-IDF** (Term Frequency–Inverse Document Frequency) mengukur seberapa penting suatu kata dalam satu dokumen relatif terhadap seluruh koleksi dokumen.

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\frac{N}{\text{DF}(t)}$$

### 3.1 TF-IDF pada Sheet Main

In [18]:
# TF-IDF vectorizer
documents = df_main['Konten'].astype(str).tolist()

vectorizer = TfidfVectorizer(
    max_features=20,
    ngram_range=(1, 2),
    stop_words=list(stop_words)
)
tfidf_matrix = vectorizer.fit_transform(documents)

# Buat DataFrame hasil TF-IDF
feature_names = vectorizer.get_feature_names_out()
tfidf_scores = tfidf_matrix.toarray()
tfidf_df = pd.DataFrame(tfidf_scores, columns=feature_names)

# Tampilkan top kata kunci berdasarkan total skor
top_keywords = tfidf_df.sum().sort_values(ascending=False).head(20)
print("✅ TF-IDF selesai dihitung!")
print("\n🔑 Top 20 Kata Kunci (Sheet Main):")
print(top_keywords.to_string())

✅ TF-IDF selesai dihitung!

🔑 Top 20 Kata Kunci (Sheet Main):
program           8.753301
mbg               8.383029
makanan           7.513167
anak              7.448690
makan             6.094812
bergizi           5.378527
sekolah           5.013915
gizi              4.743646
gratis            4.673093
makan bergizi     4.375018
siswa             4.058666
bergizi gratis    4.018241
anggaran          3.883868
juta              3.474604
000               3.401372
triliun           3.290931
pemerintah        3.163490
indonesia         2.931050
pendidikan        2.139346
guru              1.134060


### 3.2 TF-IDF pada Sheet Case

In [19]:
# TF-IDF untuk sheet Case
documents_case = df_case['Penjelasan'].astype(str).tolist()

vectorizer_case = TfidfVectorizer(
    max_features=20,
    ngram_range=(1, 2),
    stop_words=list(stop_words)
)
tfidf_matrix_case = vectorizer_case.fit_transform(documents_case)

feature_names_case = vectorizer_case.get_feature_names_out()
tfidf_scores_case = tfidf_matrix_case.toarray()
tfidf_df_case = pd.DataFrame(tfidf_scores_case, columns=feature_names_case)

top_keywords_case = tfidf_df_case.sum().sort_values(ascending=False).head(20)
print("✅ TF-IDF Sheet Case selesai!")
print("\n🔑 Top 20 Kata Kunci (Sheet Case):")
print(top_keywords_case.to_string())

✅ TF-IDF Sheet Case selesai!

🔑 Top 20 Kata Kunci (Sheet Case):
berita                2.948008
program               2.102866
makanan               1.451700
anak                  1.400082
berita program        1.222511
membahas              0.957297
berita meliput        0.941427
berita isinya         0.941427
berita menu           0.941427
ekonomi               0.889980
berita pelaksanaan    0.768462
berita membahas       0.733447
anak anak             0.700041
ahli                  0.682409
gizi                  0.622022
berita makanan        0.432480
berita fokus          0.386508
berisi                0.341205
berisi analisis       0.341205
berita berisi         0.341205


### 3.3 Perbandingan Top Keywords Antar Sheet

In [20]:
# Bandingkan top keywords dari kedua sheet
comparison = pd.DataFrame({
    'Kata Kunci (Main)': top_keywords.index.tolist()[:10],
    'Skor Main': top_keywords.values[:10].round(4),
    'Kata Kunci (Case)': top_keywords_case.index.tolist()[:10],
    'Skor Case': top_keywords_case.values[:10].round(4)
})

print("✅ Perbandingan top 10 kata kunci:")
print(comparison.to_string(index=False))

✅ Perbandingan top 10 kata kunci:
Kata Kunci (Main)  Skor Main Kata Kunci (Case)  Skor Case
          program     8.7533            berita     2.9480
              mbg     8.3830           program     2.1029
          makanan     7.5132           makanan     1.4517
             anak     7.4487              anak     1.4001
            makan     6.0948    berita program     1.2225
          bergizi     5.3785          membahas     0.9573
          sekolah     5.0139    berita meliput     0.9414
             gizi     4.7436     berita isinya     0.9414
           gratis     4.6731       berita menu     0.9414
    makan bergizi     4.3750           ekonomi     0.8900


---

## 4️⃣ Analisis N-gram

N-gram membantu mengidentifikasi frasa penting yang terdiri dari beberapa kata yang sering muncul bersama-sama.

- **Bigram** = 2 kata berurutan → contoh: *"ibu hamil"*, *"gizi buruk"*
- **Trigram** = 3 kata berurutan → contoh: *"program makan gratis"*

### 4.1 Bigram dan Trigram pada Sheet Main

In [21]:
# CountVectorizer untuk bigram dan trigram
texts_main = df_main['Konten'].dropna().tolist()

ngram_vectorizer = CountVectorizer(
    ngram_range=(2, 3),
    stop_words=list(stop_words),
    max_features=15
)
X_ngram = ngram_vectorizer.fit_transform(texts_main)

ngrams = ngram_vectorizer.get_feature_names_out()
counts = X_ngram.sum(axis=0).A1
ngram_freq = pd.DataFrame({'ngram': ngrams, 'count': counts}).sort_values(by='count', ascending=False)

print("✅ N-gram analisis selesai!")
print("\n📊 Top Bigram & Trigram (Sheet Main):")
print(ngram_freq.to_string(index=False))

✅ N-gram analisis selesai!

📊 Top Bigram & Trigram (Sheet Main):
                ngram  count
        makan bergizi     98
       bergizi gratis     93
 makan bergizi gratis     85
        program makan     54
program makan bergizi     47
            anak anak     47
          program mbg     37
   bergizi gratis mbg     31
           gratis mbg     31
     penerima manfaat     27
           badan gizi     26
     presiden prabowo     25
  anggaran pendidikan     24
        gizi nasional     24
  badan gizi nasional     23


### 4.2 Bigram dan Trigram pada Sheet Case

In [22]:
# N-gram untuk sheet Case
texts_case = df_case['Penjelasan'].dropna().tolist()

ngram_vectorizer_case = CountVectorizer(
    ngram_range=(2, 3),
    stop_words=list(stop_words),
    max_features=15
)
X_ngram_case = ngram_vectorizer_case.fit_transform(texts_case)

ngrams_case = ngram_vectorizer_case.get_feature_names_out()
counts_case = X_ngram_case.sum(axis=0).A1
ngram_freq_case = pd.DataFrame({'ngram': ngrams_case, 'count': counts_case}).sort_values(by='count', ascending=False)

print("✅ N-gram Sheet Case selesai!")
print("\n📊 Top Bigram & Trigram (Sheet Case):")
print(ngram_freq_case.to_string(index=False))

✅ N-gram Sheet Case selesai!

📊 Top Bigram & Trigram (Sheet Case):
                   ngram  count
               anak anak      2
          berita program      2
       ahli ekonomi ahli      1
               ahli gizi      1
      analisis pandangan      1
analisis pandangan pakar      1
            ahli ekonomi      1
         berisi analisis      1
           berita berisi      1
            berita fokus      1
  berita berisi analisis      1
   berita fokus membahas      1
           berita isinya      1
berita isinya pernyataan      1
          berita makanan      1


### 4.3 Export Hasil ke Excel

In [24]:
# Export hasil ke Excel
output_file = 'Dataset MBG_Keyword_Extraction.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    top_keywords.reset_index().rename(columns={'index': 'keyword', 0: 'score'}).to_excel(
        writer, sheet_name='TFIDF_Main', index=False
    )
    top_keywords_case.reset_index().rename(columns={'index': 'keyword', 0: 'score'}).to_excel(
        writer, sheet_name='TFIDF_Case', index=False
    )
    ngram_freq.to_excel(writer, sheet_name='Ngram_Main', index=False)
    ngram_freq_case.to_excel(writer, sheet_name='Ngram_Case', index=False)

print(f"✅ Hasil diekspor ke '{output_file}'")

✅ Hasil diekspor ke 'Dataset MBG_Keyword_Extraction.xlsx'


---

## 📝 Kesimpulan

- 🎯 **TF-IDF** berhasil mengidentifikasi kata kunci dominan dari artikel MBG; kata-kata seperti *program*, *mbg*, *makanan*, *anak*, dan *gizi* muncul paling signifikan.
- 📊 **Sheet Case** (artikel insiden) menampilkan kata kunci berbeda, seperti *racun*, *basi*, *keracunan* — mencerminkan isu keamanan pangan.
- 🔧 **N-gram** (bigram/trigram) membantu mengungkap frasa penting seperti *"ibu hamil"*, *"gizi buruk"*, *"program makan gratis"* yang tidak terlihat dari analisis unigram saja.
- 📂 Seluruh hasil diekspor ke file Excel untuk analisis lebih lanjut.

---

## 📚 Referensi

- Salton, G., & Buckley, C. (1988). Term-weighting approaches in automatic text retrieval. *Information Processing & Management*, 24(5), 513–523.
- Scikit-learn Documentation: [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- NLTK Documentation: [stopwords](https://www.nltk.org/)

---